#  关于模型调用的invoke
## 1. invoke的传参
### 1.1 文本输入

In [2]:
import os
from urllib import response

from dns import message
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage

# 从 .env 文件中加载环境变量
load_dotenv(override=True)

# 从环境变量读取配置
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(model="deepseek:deepseek-v4-flash",
                        #model_provider="deepseek",
                        api_key=DEEPSEEK_API_KEY,
                        base_url=DEEPSEEK_BASE_URL)


In [6]:

print(model.invoke("翻译如下汉字:你好世界"))

content='Hello, world!' additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答用户。用户说“翻译如下汉字:你好世界”。需要翻译成什么语言？没有指定，可能英文。需要给出翻译。中文“你好世界”通常英文 Hello, world. 可以简洁。需要翻译如下汉字。回答：Hello, world. 可能也可以说 Hello world! 用自然。最终只给翻译。'} response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 36, 'total_tokens': 113, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 72, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 36}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'a75d18fd-7496-4c70-9439-42e78615e929', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a1a72d-ac26-769b-8e1d-b4aeb7f5b52c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 36, 'output_tokens': 77, 'total_tokens': 113, 'input_token_de

### 1.2字典列表
多轮对话

In [9]:
message = [
      {"role": "system", "content": "你是专业的数学老师"},
      {"role": "user", "content": "帮我解释下斐波拉契数列"},
      {"role": "assistant", "content": "每一项都等于前两项之和用递推式表示"},
      {"role": "user", "content": "我刚才问了什么问题"},

]
response = model.invoke(message)
print(response)

content='你刚才问的是：\n\n**“帮我解释下斐波拉契数列”**（也就是斐波那契数列）。' additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答用户“我刚才问了什么问题”。需要回顾对话。用户之前问“帮我解释下斐波拉契数列”。然后助手回复了关于斐波那契数列的解释（但当前消息里只显示了用户那条和助手回复不完整？对话是：user: 帮我解释下斐波拉契数列。assistant: 每一项都等于前两项之和用递推式表示。现在user: 我刚才问了什么问题。需要直接回答他刚才问的是“帮我解释下斐波拉契数列”。可以简洁。注意可能指出拼写斐波拉契/斐波那契。回答中文。'} response_metadata={'token_usage': {'completion_tokens': 154, 'prompt_tokens': 60, 'total_tokens': 214, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 128, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 60}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '8e90a333-3ad2-43f0-836a-efbe596ec232', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a1a735-f5f1-7416-92e2-84696254ef96-0' tool_calls=[] invalid_tool_call

**举例 3：** 如果不传递历史， AI 会 " 失忆 "

In [2]:
import os
from urllib import response

from dns import message
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

# 从 .env 文件中加载环境变量
load_dotenv(override=True)

# 从环境变量读取配置
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(model="deepseek:deepseek-v4-flash",
                        #model_provider="deepseek",
                        api_key=DEEPSEEK_API_KEY,
                        base_url=DEEPSEEK_BASE_URL)

messages1 = [
      {"role": "system", "content": " 你是一个非常友好的 AI 助手 "},
      {"role": "user", "content": " 你好，我叫小明 "},
]
# 第一次对话
response1 = model.invoke(messages1)
# 打印响应
print(f"AI 的回复 1 ： {response1.content}")

messages2 = [
      {"role": "user", "content": " 我叫什么名字？ "}
]
# 第二次对话
response2 = model.invoke(messages2)
# 打印响应
print(f"AI 的回复 2 ： {response2.content}")


AI 的回复 1 ： 你好呀，小明！很高兴认识你 😊 我是你的 AI 助手，有什么想聊的、想问的，或者需要帮忙的，随时告诉我就好～
AI 的回复 2 ： 我不知道你的名字——我们的对话里没有提供。你可以告诉我希望我怎么称呼你。


作为对比，传递记忆：

In [3]:
conversation = [
      {"role": "system", "content": " 你是一个非常友好的 AI 助手 "},
      {"role": "user", "content": " 你好，我叫小明 "}
]

# 第一次对话
response1 = model.invoke(conversation)
# 打印响应
print(f"AI 的回复 1 ： {response1.content}")

# 添加记忆
conversation.append({"role": "assistant", "content": response1.content})

conversation.append({"role": "user", "content": " 我叫什么名字？ "})

# 第二次对话
response2 = model.invoke(conversation)
print(f"AI 的回复 2 ： {response2.content}")

AI 的回复 1 ： 你好，小明！很高兴认识你 😊 有什么我可以帮你的吗？
AI 的回复 2 ： 你叫小明呀 😊


##  1.3消息对象列表

In [4]:
import os

from langchain_core.messages import SystemMessage, AIMessage, HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

# 从 .env 文件中加载环境变量
load_dotenv(override=True)

# 从环境变量读取配置
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(model="deepseek:deepseek-v4-flash",
                        #model_provider="deepseek",
                        api_key=DEEPSEEK_API_KEY,
                        base_url=DEEPSEEK_BASE_URL)

# 使用消息对象格式构建消息
messages = [
      SystemMessage(" 你是一个专业的数学老师。 "),
      HumanMessage("2 + 3 * 2 = ？ "),
      AIMessage("8"),
      HumanMessage(" 我刚才问什么问题了？ ")
]

response = model.invoke(messages)
# 打印响应
print(f"AI 的回复： {response.content}")

AI 的回复： 你刚才问的是：**“2 + 3 * 2 = ？”**  
按运算顺序，答案是 **8**。


## **举例 2：**

In [10]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

messages = [
      SystemMessage(content=" 你是一个 Python 专家 "),
      HumanMessage(content=" 什么是生成器？ "),
]

response = model.invoke(messages)
# print(response)

# 继续对话
messages.append(AIMessage(content=response.content))
messages.append(HumanMessage(content=" 能给个例子吗？ "))

response1 = model.invoke(messages)
print(response1)

content='# 生成器实例：从「看懂流程」到「实际用途」\n\n下面给几个由浅入深的例子，都能直接运行。\n\n---\n\n## 例 1：最小示例 —— 看清楚 `yield` 的暂停/恢复\n\n```python\ndef countdown(n):\n    print("  [开始执行]")\n    while n > 0:\n        print(f"  [准备交出 {n}]")\n        yield n          # 暂停：把 n 交给调用者，保存现场\n        print(f"  [从 yield 处恢复，n={n}]")\n        n -= 1\n    print("  [执行结束]")\n\nprint(">>> 创建生成器（注意：函数体没有执行）")\ng = countdown(3)\n\nprint(">>> 第一次 next()")\nprint("拿到:", next(g))\n\nprint(">>> 第二次 next()")\nprint("拿到:", next(g))\n\nprint(">>> 用 for 循环把剩下的取完")\nfor v in g:\n    print("拿到:", v)\n```\n\n输出：\n\n```\n>>> 创建生成器（注意：函数体没有执行）\n>>> 第一次 next()\n  [开始执行]\n  [准备交出 3]\n拿到: 3\n>>> 第二次 next()\n  [从 yield 处恢复，n=3]\n  [准备交出 2]\n拿到: 2\n>>> 用 for 循环把剩下的取完\n  [从 yield 处恢复，n=2]\n  [准备交出 1]\n拿到: 1\n  [从 yield 处恢复，n=1]\n  [执行结束]\n```\n\n**关键点**：`countdown(3)` 那一刻函数体一行都没跑，只是返回了一个生成器对象。真正的执行被 `next()` 一次次「推进」。\n\n---\n\n## 例 2：实用场景 —— 处理一个几十 G 的大文件\n\n假设有一个巨大日志文件，每一行是 `时间,用户ID,金额`，我们要统计金额 > 100 的记录条数。\n\n### ❌ 不用生成器（会把文件全读进内存）\n\n```python\nwith

##  invoke 返回一个 AIMessage 对象

In [11]:
# 使用字典格式构建消息
from rich import print as rprint

response = model.invoke([HumanMessage("2 + 3 * 2 = ？ ")])
# 打印响应
rprint(response)
# <class 'langchain_core.messages.ai.AIMessage'>

AIMessage(
    content='8',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': 'We need answer. Need consider standard order operations: 3*2=6 +2=8. User in Chinese 
asks. Need concise.'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 31,
            'prompt_tokens': 41,
            'total_tokens': 72,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 29,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 41
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6',
        'id': 'b72b3ceb-0fe9-4e5a-a4c9-fb598af6cc7c',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a1b007-6b13-755a-8537-1d722443c192-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 41,
        'output_tokens': 31,
        'total_tokens': 72,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 29}
    }
)

## 1.4stream流式调用

In [12]:
import os

from langchain_core.messages import SystemMessage, AIMessage, HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

# 从 .env 文件中加载环境变量
load_dotenv(override=True)

# 从环境变量读取配置
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(model="deepseek:deepseek-v4-flash",
                        #model_provider="deepseek",
                        api_key=DEEPSEEK_API_KEY,
                        base_url=DEEPSEEK_BASE_URL)

for chunk in model.stream(" 写一首七言律诗，总结大模型的发展 "):
      print(chunk.text, end="", flush=True)  # 逐 token 输出

《七律·大模型发展》

算力连云数据洪，  
模型万亿势如虹。  
深层网络开新境，  
预训宏开不世功。  
定律规模催跃起，  
多模态汇百科通。  
人机价值须同守，  
百业同春沐好风。

举例1 batch的使用

In [ ]:
import os

from langchain_core.messages import SystemMessage, AIMessage, HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint

# 从 .env 文件中加载环境变量
load_dotenv(override=True)

# 从环境变量读取配置
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(model="deepseek:deepseek-v4-flash",
                        #model_provider="deepseek",
                        api_key=DEEPSEEK_API_KEY,
                        base_url=DEEPSEEK_BASE_URL)

messages = [

      " 你好，你是谁？ ",
      "2 + 3 * 5 = ?",
      " 中国首都在哪里？ "

]
#model.batch() 要求传入的是一个列表
responses = model.batch(messages)
rprint(responses)


举例2 batch_as_completed的使用

In [19]:
from langchain_core.messages import SystemMessage, AIMessage, HumanMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rprint

# 从 .env 文件中加载环境变量
load_dotenv(override=True)

# 从环境变量读取配置
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(model="deepseek:deepseek-v4-flash",
                        #model_provider="deepseek",
                        api_key=DEEPSEEK_API_KEY,
                        base_url=DEEPSEEK_BASE_URL)

messages = [
      " 你好，你是谁？ ",
      "2 + 3 * 5 = ?",
      " 中国首都在哪里？ "
]

responses = model.batch_as_completed(messages)

for response in responses:
      print(response)

(1, AIMessage(content='17', additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer simple. Need consider order ops = 2 + 15=17. Need concise.'}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 39, 'total_tokens': 62, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 21, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 39}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': 'ef6cd264-9519-465e-ab32-7a424706421c', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1b019-97f0-7313-81f2-d8f4133069df-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 39, 'output_tokens': 23, 'total_tokens': 62, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning

## 1.5 异步调用


In [20]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
import asyncio
import time

# 从 .env 文件中加载环境变量
load_dotenv(override=True)

# 从环境变量读取配置
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(model="deepseek:deepseek-v4-flash",
                        #model_provider="deepseek",
                        api_key=DEEPSEEK_API_KEY,
                        base_url=DEEPSEEK_BASE_URL)

In [27]:
async def demo_async_invoke():
      print("=== 演示： ainvoke 的异步（非阻塞）效果 ===")


start_time = time.perf_counter()  # 记录开始时间

print(" 程序开始 ...")
print(">>> 发起异步模型调用 (ainvoke)...")
async_task = asyncio.create_task(model.ainvoke(" 用一句话解释人工智能。 "))

# 2. 并行执行其他任务
print(">>> 模型请求已在后台发送，继续执行本地逻辑 ...")
for i in range(3):
      await asyncio.sleep(1)  # 使用异步等待，释放控制权
print(f">>> 正在执行第 {i + 1} 个任务 ... ( 已耗时 {time.perf_counter() -
                                                    start_time:.2f}s)")

# 3. 获取模型结果
print(">>> 本地任务完成，检查模型状态 ...")
response = await async_task

end_time = time.perf_counter()
print(f">>> 模型返回 : {response.content}")
#计算从开始到结束经过的秒数（浮点数 ）如果是变量名就是不是减法end_time - start_time
print(f"=== 总运行耗时 : {end_time - start_time:.2f}s ===")


async def main():
      """ 主函数 """


await demo_async_invoke()

# if __name__ == "__main__":
#       asyncio.run(main())

 程序开始 ...
>>> 发起异步模型调用 (ainvoke)...
>>> 模型请求已在后台发送，继续执行本地逻辑 ...
>>> 正在执行第 3 个任务 ... ( 已耗时 3.02s)
>>> 本地任务完成，检查模型状态 ...
>>> 模型返回 : 人工智能是让机器模拟、延伸和扩展人类智能，从而具备感知、学习、推理和决策能力来完成复杂任务的技术。
=== 总运行耗时 : 3.02s ===
=== 演示： ainvoke 的异步（非阻塞）效果 ===


举例 2： astream()

In [32]:
async def demo_async_stream():
      """ 演示异步调用的非阻塞特性 """


print("=== 演示： astream 的异步（非阻塞）效果 ===")
start_time = time.perf_counter()  # 记录开始时间
print(" 程序开始 ...")

# 1. 发起异步流式请求
# 注意：此时请求已发出，返回的是一个异步生成器
print(">>> 发起异步流式调用 (astream)...")
stream_resp = model.astream(" 请用一句话解释机器学习的基本概念。 ")

# 2. 在等待流式响应的同时，执行其他任务
print(">>> 流式请求已发送，程序无需等待，继续执行其他异步任务 ...")
for i in range(3):
      # 使用 asyncio.sleep 而非 time.sleep
      # 这允许事件循环在等待时去处理上面的 stream_resp 网络 IO
      await asyncio.sleep(1)
# print(f">>> 正在执行并发任务 {i + 1}... ")
print(f">>> 正在执行第 {i + 1} 个任务 ... ( 已耗时 {time.perf_counter() -
                                                    start_time:.2f}s)")

# 3. 现在开始处理流式结果
print(">>> 模拟任务已完成，开始读取缓冲区中的流式结果 ...")
end_time = time.perf_counter()
print(">>> 流式输出 : ", end="", flush=True)
async for chunk in stream_resp:
      # LangChain 的消息块通常通过 .content 获取内容
      content = chunk.content if hasattr(chunk, 'content') else str(chunk)
print(content, end="", flush=True)

print("\n>>> 流式输出结束 \n")
print(f"=== 总运行耗时 : {end_time - start_time:.2f}s ===")


async def main():
      """ 主函数 """


await demo_async_stream()

# if __name__ == "__main__":
#       asyncio.run(main())

=== 演示： astream 的异步（非阻塞）效果 ===
 程序开始 ...
>>> 发起异步流式调用 (astream)...
>>> 流式请求已发送，程序无需等待，继续执行其他异步任务 ...
>>> 正在执行第 3 个任务 ... ( 已耗时 3.03s)
>>> 模拟任务已完成，开始读取缓冲区中的流式结果 ...
>>> 流式输出 : 
>>> 流式输出结束 

=== 总运行耗时 : 3.03s ===
